# Proyecto 2 – Análisis Exploratorio
### Data Science

**Reto 18: Hackeando el cuerpo humano** — [HuBMAP - Hacking the Human Body](https://www.kaggle.com/competitions/hubmap-organ-segmentation)

**División del trabajo (secciones consecutivas, no intercaladas):**
- Persona 1 → secciones 1, 2, 3 y 4.a
- Persona 2 → secciones 4.b y 4.c
- Persona 3 → secciones 4.d y 5


---
## Configuración inicial

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_DIR = "data/"

---
## 1. Investigación del tema

HuBMAP es un programa que busca mapear el cuerpo humano a nivel de célula, algo así como armar un mapa detallado de cómo están organizados los tejidos sanos en cada órgano. La competencia de Kaggle nace de ahí y junta datos de HuBMAP con los del Atlas de Proteínas Humanas (HPA), y lo que pide es encontrar y delimitar, dentro de fotos de tejido, las llamadas unidades funcionales de tejido (FTU).

Una FTU es básicamente un grupo pequeño de células organizadas alrededor de un vaso sanguíneo, que en conjunto cumplen una función específica dentro del órgano al que pertenecen. El reto incluye cinco órganos: riñón, próstata, bazo, pulmón e intestino grueso, y en cada uno la FTU se ve distinta porque cada órgano tiene su propia estructura.

Las imágenes vienen de cortes de tejido que se tiñen con una tinción llamada PAS, que resalta ciertas estructuras del tejido para que se puedan distinguir mejor al verlas al microscopio. Como los datos vienen de dos fuentes distintas (HuBMAP y HPA), y cada una prepara y escanea las muestras a su manera, un mismo órgano puede verse un poco diferente dependiendo de dónde salió la imagen.

Todo esto importa porque la idea final es lograr que un modelo reconozca estas unidades de tejido sin importar el órgano ni la fuente de la imagen, y no solo memorice los casos que vio en el entrenamiento. Eso es justo lo que hace difícil el problema, y es la base para poder construir ese mapa completo del cuerpo humano que se mencionaba al principio.

---
## 2. Análisis del problema planteado y los datos

### Situación problemática

Hoy en día identificar las unidades funcionales de tejido en una imagen de biopsia lo hacen patólogos a mano, revisando la muestra al microscopio y marcando dónde están esas estructuras. Esto toma tiempo, depende de la experiencia de la persona que lo hace y no es algo que se pueda escalar a miles de imágenes de distintos órganos. Además, como las muestras pueden venir de laboratorios distintos con procesos de preparación distintos, dos imágenes del mismo órgano no siempre se ven igual, lo que complica todavía más hacer este trabajo de forma manual y consistente.

### Problema científico

El problema se puede plantear como una pregunta de segmentación de imágenes: a partir de una imagen de tejido, ¿se puede entrenar un modelo que marque con precisión en qué zonas de la imagen están las unidades funcionales de tejido, sin importar de qué órgano se trate ni de qué fuente venga la muestra? La dificultad extra es que el dataset solo trae unos cuantos órganos y fuentes de datos, entonces el modelo tiene que aprender un patrón que generalice y no simplemente memorizar cómo se ve cada órgano en particular.

### Objetivos

**Objetivo general:**

Explorar y describir los datos de la competencia HuBMAP para entender cómo están compuestas las imágenes de tejido y las variables que las acompañan, de forma que sirva de base para pensar en una futura solución al problema de segmentación de FTU.

**Objetivos específicos:**
1. Describir las variables numéricas y categóricas del dataset (órgano, fuente de datos, edad, sexo, grosor del tejido, dimensiones de imagen) y detectar diferencias entre grupos.
2. Analizar cómo se relacionan variables como el órgano, la fuente de datos, la edad y el sexo, para identificar qué tan variadas son las imágenes según esos factores.

### Descripción general de los datos

In [4]:
train = pd.read_csv(DATA_DIR + "train.csv")

print("train:", train.shape)
train.head()

train: (351, 10)


,id,organ,data_source,img_height,img_width,pixel_size,tissue_thickness,rle,age,sex
0,10044,prostate,HPA,3000,3000,0.4,4,1459676 77 1462675 82 1465674 87 1468673 92 14...,37.0,Male
1,10274,prostate,HPA,3000,3000,0.4,4,715707 2 718705 8 721703 11 724701 18 727692 3...,76.0,Male
2,10392,spleen,HPA,3000,3000,0.4,4,1228631 20 1231629 24 1234624 40 1237623 47 12...,82.0,Male
3,10488,lung,HPA,3000,3000,0.4,4,3446519 15 3449517 17 3452514 20 3455510 24 34...,78.0,Male
4,10610,spleen,HPA,3000,3000,0.4,4,478925 68 481909 87 484893 105 487863 154 4908...,21.0,Female


El archivo train.csv tiene 351 filas y 10 columnas, cada fila es una imagen de tejido con su información asociada: el órgano al que pertenece, si la muestra viene de HuBMAP o de HPA, el alto y ancho de la imagen, el tamaño de píxel, el grosor del tejido, la máscara de la FTU codificada en formato rle, y la edad y sexo del donante. Solo se usa este archivo porque es el único que trae datos completos, ya que no se va a participar en la competencia como tal.

---
## 3. Limpieza y preprocesamiento

In [5]:
train.isna().sum()

id                  0
organ               0
data_source         0
img_height          0
img_width           0
pixel_size          0
tissue_thickness    0
rle                 0
age                 0
sex                 0
dtype: int64

In [6]:
print("Duplicados en train:", train.duplicated().sum())

Duplicados en train: 0


In [7]:
for col in ["organ", "data_source", "sex"]:
    train[col] = train[col].astype("category")

train.dtypes

id                     int64
organ               category
data_source         category
img_height             int64
img_width              int64
pixel_size           float64
tissue_thickness       int64
rle                   object
age                  float64
sex                 category
dtype: object

No se encontraron valores nulos ni filas duplicadas en train.csv, así que no hubo que eliminar ni imputar nada. Lo único que se hizo fue convertir organ, data_source y sex a tipo categórico, porque pandas las traía como texto plano y así quedan mejor identificadas para el análisis. La columna rle no se decodifica aquí para todo el dataset porque son máscaras pesadas, esa decodificación se hace más adelante, solo sobre la muestra de imágenes que se use para la parte visual.

---
## 4. Análisis exploratorio de los datos

### 4.a Variables y observaciones disponibles

In [8]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 351 entries, 0 to 350
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   id                351 non-null    int64   
 1   organ             351 non-null    category
 2   data_source       351 non-null    category
 3   img_height        351 non-null    int64   
 4   img_width         351 non-null    int64   
 5   pixel_size        351 non-null    float64 
 6   tissue_thickness  351 non-null    int64   
 7   rle               351 non-null    object  
 8   age               351 non-null    float64 
 9   sex               351 non-null    category
dtypes: category(3), float64(2), int64(4), object(1)
memory usage: 20.8+ KB


En total train.csv tiene 351 observaciones (una por cada imagen de tejido) y 10 variables. id es solo el identificador de la imagen, no aporta nada analítico por sí solo. organ, data_source y sex son categóricas, ya convertidas a ese tipo en la sección anterior: organ tiene los cinco órganos del reto, data_source indica si la muestra viene de HuBMAP o de HPA, y sex es el sexo del donante. img_height e img_width son numéricas y dan el alto y ancho de la imagen en píxeles. pixel_size y tissue_thickness también son numéricas y describen el tamaño real que representa cada píxel y el grosor de la muestra de tejido. age es numérica y es la edad del donante. Por último, rle es una variable de texto que guarda la máscara de la FTU codificada, no es un valor que se pueda resumir como las demás.

### 4.b Resumen de variables numéricas y tablas de frecuencia de categóricas

#### Resumen de variables numéricas

In [7]:
train.describe()

,id,img_height,img_width,pixel_size,tissue_thickness,age
count,351.000000,351.000000,351.000000,3.510000e+02,351.0,351.000000
mean,16662.914530,2978.364672,2978.364672,4.000000e-01,4.0,60.364672
std,9863.945557,90.962085,90.962085,1.111808e-16,0.0,16.013327
min,62.000000,2308.000000,2308.000000,4.000000e-01,4.0,21.000000
25%,8229.000000,3000.000000,3000.000000,4.000000e-01,4.0,55.000000
50%,16609.000000,3000.000000,3000.000000,4.000000e-01,4.0,60.000000
75%,25630.500000,3000.000000,3000.000000,4.000000e-01,4.0,73.000000
max,32741.000000,3070.000000,3070.000000,4.000000e-01,4.0,84.000000


Casi todas las imágenes miden 3000x3000 píxeles: la mediana y el percentil 75 de `img_height` e `img_width` caen justo en 3000, y el mínimo (2308) muestra que hay algunas imágenes más pequeñas, pero la variación es baja frente al tamaño típico (desviación estándar ≈ 91 píxeles). `pixel_size` y `tissue_thickness` no varían nada (0.4 y 4 en las 351 filas, desviación estándar = 0), así que en este archivo no aportan información para diferenciar observaciones. Esto se explica porque, como se ve en la tabla de frecuencias de abajo, `train.csv` solo trae imágenes de origen HPA, que siempre se escanean con esos mismos parámetros; en el set de HuBMAP real estos valores sí varían. `id` no se interpreta como variable numérica real, es solo un identificador.

En cuanto a `age`, hay bastante dispersión: el promedio es de 60 años, con donantes de entre 21 y 84 años y una desviación estándar de 16 años, lo que indica que el dataset cubre un rango amplio de edades adultas.

#### Tablas de frecuencia de variables categóricas

In [8]:
for col in ["organ", "data_source", "sex"]:
    if col in train.columns:
        print(f"\n--- {col} ---")
        print(train[col].value_counts())
        print(train[col].value_counts(normalize=True).round(3))


--- organ ---
organ
kidney            99
prostate          93
largeintestine    58
spleen            53
lung              48
Name: count, dtype: int64
organ
kidney            0.282
prostate          0.265
largeintestine    0.165
spleen            0.151
lung              0.137
Name: proportion, dtype: float64

--- data_source ---
data_source
HPA    351
Name: count, dtype: int64
data_source
HPA    1.0
Name: proportion, dtype: float64

--- sex ---
sex
Male      229
Female    122
Name: count, dtype: int64
sex
Male      0.652
Female    0.348
Name: proportion, dtype: float64


En `organ`, kidney (99, 28.2%) y prostate (93, 26.5%) son los órganos con más observaciones, seguidos de largeintestine (58, 16.5%), spleen (53, 15.1%) y lung (48, 13.7%, el menos representado). No es un desbalance extremo, pero conviene tenerlo en cuenta si más adelante se comparan métricas entre órganos.

En `data_source`, el 100% de las filas vienen de HPA, no hay ninguna observación de HuBMAP en `train.csv`. Este es un hallazgo importante: aunque la competencia combina ambas fuentes, el archivo con el que se está trabajando solo trae la parte pública de HPA (el resto de HuBMAP probablemente se queda en el set de prueba oculto de la competencia), así que este análisis describe solo esa parte de los datos, no el problema completo.

En `sex`, el 65.2% de los donantes son hombres (229) y el 34.8% mujeres (122). Este desbalance se explica en parte por `prostate`, que por razones biológicas solo tiene donantes hombres, como se revisa en la sección 4.c.

### 4.c Cruce de variables importantes

#### Cruce de variables clave

_Completar: ej. órgano vs. edad/sexo/grosor de tejido._

#### Correlaciones entre variables numéricas

In [ ]:
numeric_cols = train.select_dtypes(include="number").columns
train[numeric_cols].corr()

#### Outliers y valores faltantes

_Completar: identificación de outliers (ej. boxplots por órgano), explicación de posibles causas, y decisión tomada ante valores faltantes (si los hay)._

---
### 4.d Gráficos exploratorios

#### Variable derivada: área de la FTU

Antes de graficar se construye una variable extra a partir de la columna `rle`: la cantidad de píxeles
que ocupa la máscara de la FTU en cada imagen (`mask_px`) y esa misma cantidad como proporción del
área total de la imagen (`mask_frac`). No hace falta decodificar la máscara completa para esto: el
formato run-length guarda pares (inicio, largo), así que basta con sumar los largos. Esta variable es
la que más se acerca a lo que el modelo tendría que predecir, por eso vale la pena incluirla en el
análisis exploratorio.

In [ ]:
def rle_area(rle_string):
    """Cuenta los píxeles marcados en una máscara rle sin decodificarla por completo."""
    lengths = np.asarray(rle_string.split()[1::2], dtype=int)
    return int(lengths.sum())


train["mask_px"] = train["rle"].map(rle_area)
train["mask_frac"] = train["mask_px"] / (train["img_height"] * train["img_width"])

train[["organ", "img_height", "img_width", "mask_px", "mask_frac"]].head()

#### Gráficos univariados

In [ ]:
num_cols = ["age", "img_height", "mask_frac"]
titulos = ["Edad del donante", "Alto de la imagen (px)", "Proporción de la imagen ocupada por la FTU"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for i, (col, titulo) in enumerate(zip(num_cols, titulos)):
    sns.histplot(train[col], bins=25, ax=axes[0, i], color="#4C72B0")
    axes[0, i].set_title(titulo)
    axes[0, i].set_xlabel("")

    sns.boxplot(x=train[col], ax=axes[1, i], color="#4C72B0")
    axes[1, i].set_xlabel(titulo)

plt.tight_layout()
plt.show()

En edad la distribución va de 21 a 84 años, se concentra entre los 55 y los 73 y tiene una mediana
de 60 años. Es una población de donantes mayoritariamente adulta-mayor y la cola hacia los donantes
jóvenes es bastante más delgada, así que el dataset no representa por igual todas las edades.

En alto de imagen el gráfico se ve casi como una sola barra: 326 de las 351 imágenes son de
3000 × 3000 píxeles y el resto son un poco más chicas (el mínimo es 2308). El boxplot marca todas
esas imágenes menores como outliers, pero no son errores: simplemente son recortes de tejido que no
alcanzaron el tamaño estándar. De hecho en las 351 filas el alto y el ancho siempre son iguales, o sea
que todas las imágenes son cuadradas.

En proporción de FTU la distribución está claramente sesgada a la derecha: la mediana es 0.059
(alrededor del 6 % de la imagen) pero hay casos que llegan a 0.43. Esto quiere decir que en la mayoría
de las imágenes la estructura a segmentar ocupa una parte pequeña de la foto, lo cual es información
relevante porque implica un problema con clases desbalanceadas a nivel de píxel.

In [ ]:
cat_cols = ["organ", "sex", "data_source"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, col in zip(axes, cat_cols):
    orden = train[col].value_counts().index
    sns.countplot(data=train, x=col, order=orden, ax=ax, hue=col, legend=False, palette="deep")
    ax.set_title(f"Frecuencia de {col}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

En organ las cinco clases están desbalanceadas: riñón (99) y próstata (93) juntas son más de la
mitad del dataset, mientras que pulmón (48) y bazo (53) son las menos representadas. Un modelo
entrenado con esto tendría bastante menos ejemplos para aprender cómo se ve una FTU de pulmón.

En sex hay 229 donantes masculinos contra 122 femeninos, casi dos a uno.

En data_source el gráfico muestra una sola barra: las 351 filas de `train.csv` vienen de HPA. Las
muestras de HuBMAP están del lado de los datos de prueba de la competencia. Esto es un hallazgo
importante y no es un detalle menor: significa que el set de entrenamiento es homogéneo en cuanto a
fuente, y que la variabilidad entre laboratorios —que es justo la dificultad principal del reto— no se
puede observar dentro de estos datos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

orden_organ = train["organ"].value_counts().index

sns.boxplot(data=train, x="organ", y="mask_frac", order=orden_organ, ax=axes[0],
            hue="organ", legend=False, palette="deep")
axes[0].set_title("Proporción de FTU por órgano")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=30)

sns.boxplot(data=train, x="organ", y="age", order=orden_organ, ax=axes[1],
            hue="organ", legend=False, palette="deep")
axes[1].set_title("Edad del donante por órgano")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=30)

tabla = pd.crosstab(train["organ"], train["sex"], normalize="index").loc[orden_organ]
tabla.plot(kind="bar", stacked=True, ax=axes[2], colormap="Set2")
axes[2].set_title("Composición por sexo dentro de cada órgano")
axes[2].set_xlabel("")
axes[2].set_ylabel("proporción")
axes[2].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

El primer gráfico es el más informativo de los tres. La proporción de imagen ocupada por la FTU cambia
muchísimo según el órgano: en intestino grueso la mediana es de casi 0.20 y en próstata de 0.14,
mientras que en pulmón apenas llega a 0.013 y en riñón a 0.028. Es decir, las FTU de pulmón ocupan más
o menos quince veces menos superficie que las de intestino grueso. Además la dispersión también es
distinta: próstata y bazo tienen cajas muy anchas y intestino grueso una más compacta. Esto confirma
que "encontrar la FTU" no es el mismo problema en cada órgano.

En edad por órgano destaca intestino grueso, con una mediana de 83 años frente a 57-59 en los
otros cuatro órganos. No es un error de datos sino cómo están repartidos los donantes por tipo de
muestra, pero conviene tenerlo presente: órgano y edad no son independientes en este dataset.

En sexo por órgano se ve el caso extremo de próstata, que es 100 % masculino por razones
biológicas obvias. Los otros cuatro órganos están bastante más repartidos, aunque riñón se inclina
hacia donantes masculinos. Como próstata es el segundo órgano más frecuente, buena parte del
desbalance general hacia donantes masculinos se explica por ahí.

#### EDA visual de imágenes

Para esta parte se trabaja sobre una muestra chica de `train_images/` (no sobre todo el set, que pesa
varios GB). Se toma una imagen por órgano, se decodifica su máscara desde la columna `rle` y se
sobrepone sobre el tejido para ver qué es lo que en realidad hay que segmentar.

Si la carpeta `data/train_images/` no existe, las celdas siguientes avisan y no fallan.

In [ ]:
def rle_decode(rle_string, shape):
    """Decodifica una máscara en formato run-length encoding a un array binario (h, w)."""
    s = rle_string.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths
    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        mask[lo:hi] = 1
    return mask.reshape(shape)

In [ ]:
import os
from PIL import Image

IMG_DIR = DATA_DIR + "train_images/"

if not os.path.isdir(IMG_DIR):
    print(f"No se encontró {IMG_DIR}. Descargar una muestra de train_images/ para ver esta parte.")
    disponibles = pd.DataFrame()
else:
    ids_locales = {int(f.split(".")[0]) for f in os.listdir(IMG_DIR) if f.endswith(".tiff")}
    disponibles = train[train["id"].isin(ids_locales)]
    print(f"Imágenes disponibles localmente: {len(disponibles)} de {len(train)}")
    print(disponibles["organ"].value_counts().to_string())

In [ ]:
if len(disponibles) > 0:
    # una imagen por órgano, hasta donde alcance la muestra descargada
    muestra = disponibles.groupby("organ", observed=True).head(1)
    n = len(muestra)

    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8.5), squeeze=False)
    for j, (_, fila) in enumerate(muestra.iterrows()):
        img = np.array(Image.open(f"{IMG_DIR}{fila['id']}.tiff"))
        mask = rle_decode(fila["rle"], (fila["img_height"], fila["img_width"]))

        axes[0, j].imshow(img)
        axes[0, j].set_title(f"{fila['organ']} · id {fila['id']}\n{img.shape[1]}×{img.shape[0]} px")
        axes[0, j].axis("off")

        axes[1, j].imshow(img)
        axes[1, j].imshow(mask, alpha=0.35, cmap="autumn")
        axes[1, j].set_title(f"FTU: {fila['mask_frac']:.1%} de la imagen")
        axes[1, j].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("Sin imágenes locales: no se genera el gráfico.")

Viendo las imágenes se entiende mejor lo que los números ya insinuaban. Todas las muestras son cortes
de tejido teñidos con PAS, con tonos rosados y morados bastante parecidos entre sí, y en todas hay
zonas de fondo blanco donde no hay tejido. La máscara superpuesta muestra que las FTU no son una sola
región compacta: en la mayoría de los órganos son varias regiones separadas dentro de la misma imagen
(glomérulos en riñón, criptas en intestino grueso, glándulas en próstata), y su forma cambia por
completo de un órgano a otro.

También se nota que el contraste entre la FTU y el tejido de alrededor no es igual en todos lados: en
intestino grueso y próstata las estructuras se distinguen a simple vista, mientras que en pulmón y
riñón la FTU se confunde mucho más con el tejido circundante y ocupa una fracción mínima de la foto.
Eso encaja con lo que mostraba el boxplot de `mask_frac` y anticipa que esos dos órganos van a ser los
más difíciles del reto.

---
## 5. Conclusiones

El análisis exploratorio de `train.csv` (351 imágenes, 10 variables, sin nulos ni duplicados) deja
varios hallazgos que vale la pena resumir.

Sobre la estructura de los datos. Tres de las variables numéricas no aportan información:
`pixel_size` vale 0.4 en las 351 filas y `tissue_thickness` vale 4.0 en todas, así que son constantes
y su varianza es cero (por eso aparecen como `NaN` en la matriz de correlación). `img_height` e
`img_width` son iguales entre sí en todas las filas —todas las imágenes son cuadradas— y en el 93 % de
los casos valen 3000. En la práctica, de las variables tabulares las únicas con información real son
`organ`, `sex`, `age` y la que se derivó de `rle`. La correlación entre edad y tamaño de imagen es
prácticamente nula (0.04), o sea que no hay relación lineal entre las pocas numéricas que varían.

Sobre el desbalance. El dataset está desbalanceado en varios ejes a la vez: por órgano (99 riñones
contra 48 pulmones), por sexo (229 hombres contra 122 mujeres, en buena parte por los 93 casos de
próstata que son todos masculinos) y por edad, con una concentración fuerte entre los 55 y 73 años.
Además, órgano y edad no son independientes: intestino grueso tiene una mediana de 83 años frente a
los 57-59 del resto.

Sobre la variable objetivo. La FTU ocupa en mediana apenas el 5.9 % de la imagen, y esa proporción
varía enormemente según el órgano: 19.6 % en intestino grueso contra 1.3 % en pulmón. Un modelo de
segmentación entrenado sobre esto enfrenta un desbalance de clases fuerte a nivel de píxel, y ese
desbalance no es uniforme entre órganos.

Sobre la generalización. El hallazgo más importante del análisis es que las 351 filas de
`train.csv` tienen `data_source = HPA`. Las muestras de HuBMAP quedan del lado de los datos de prueba.
Como la dificultad central del reto es justamente que el modelo funcione con muestras preparadas en
laboratorios distintos, resulta que la variabilidad que el modelo tiene que superar no se puede
observar ni medir dentro de los datos de entrenamiento.

Próximos pasos. A partir de esto, lo que tendría sentido hacer después es: (1) descartar
`pixel_size` y `tissue_thickness` como predictores, ya que son constantes; (2) tratar el problema como
segmentación desbalanceada, usando métricas y funciones de pérdida pensadas para eso en lugar de
exactitud por píxel; (3) validar por órgano y no solo de forma global, porque el desempeño promedio
esconde que pulmón y riñón son casos mucho más difíciles; y (4) apoyarse fuerte en aumentación de
datos de color y tinción, que es la vía razonable para simular la variabilidad entre fuentes que el
set de entrenamiento no contiene.